In [1]:
import os, glob, pathlib, getpass, time

def set_scratch_env():
    # 1) If already set, keep it
    scr = os.environ.get("SCRATCH", "").strip()
    if scr and os.path.isdir(scr):
        return scr

    # 2) Try SLURM envs (if any)
    user = os.environ.get("USER", getpass.getuser())
    jid  = os.environ.get("SLURM_JOB_ID", "").strip()
    if user and jid:
        cand = f"/scratch/{user}/{jid}"
        if os.path.isdir(cand):
            os.environ["SCRATCH"] = cand
            return cand

    # 3) Fall back: pick the most recent /scratch/<user>/* directory
    base = f"/scratch/{user}"
    paths = sorted(glob.glob(f"{base}/*"), key=lambda p: os.path.getmtime(p), reverse=True)
    for p in paths:
        try:
            test = pathlib.Path(p) / ".__write_test__"
            with open(test, "w") as f: f.write("ok")
            test.unlink()
            os.environ["SCRATCH"] = p
            return p
        except Exception:
            continue

    raise RuntimeError("Could not locate a writable /scratch/<user>/<jobid>. "
                       "Restart the kernel inside the Slurm job or choose another kernel.")

SCRATCH = set_scratch_env()
# print("SCRATCH =", SCRATCH)

In [2]:
import os, pathlib, subprocess

SCRATCH = os.environ["SCRATCH"]  # you already have it now
RUN_ROOT = os.path.join(SCRATCH, "hf_runs")
HF_HOME = os.path.join(RUN_ROOT, "hf_home")
TRANSFORMERS_CACHE = os.path.join(RUN_ROOT, "transformers_cache")
TORCH_HOME = os.path.join(RUN_ROOT, "torch_home")
for d in [RUN_ROOT, HF_HOME, TRANSFORMERS_CACHE, TORCH_HOME]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = TRANSFORMERS_CACHE
os.environ["TORCH_HOME"] = TORCH_HOME

# print("SCRATCH =", SCRATCH)
# print("RUN_ROOT =", RUN_ROOT)

In [3]:
import os, pathlib, subprocess

# Get SCRATCH from the kernel env; if missing, read from login shell
scratch = os.environ.get("SCRATCH")
if not scratch:
    scratch = subprocess.run(["bash","-lc","echo -n $SCRATCH"], capture_output=True, text=True).stdout
if not scratch:
    raise RuntimeError("SCRATCH is not set in this kernel. Start the notebook via an HPCC allocation (OnDemand).")

run_root = f"{scratch}/hf_runs"
env = {
    "HF_HOME":               f"{run_root}/hf_home",
    "HUGGINGFACE_HUB_CACHE": f"{run_root}/hf_home/hub",
    "TRANSFORMERS_CACHE":    f"{run_root}/transformers",
    "TORCH_HOME":            f"{run_root}/torch_home",
    "PIP_CACHE_DIR":         f"{run_root}/pip_cache",
}
os.environ.update(env)

# Make sure the dirs exist
for p in env.values():
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

# print("SCRATCH =", scratch)
for k, v in env.items():
    pass
    # print(f"{k} = {v}")

In [4]:
import os, subprocess, textwrap
# print({k: os.environ[k] for k in ["HF_HOME","HUGGINGFACE_HUB_CACHE","TRANSFORMERS_CACHE","TORCH_HOME","PIP_CACHE_DIR"]})
# print(subprocess.run(["bash","-lc","set -o pipefail; df -h \"$SCRATCH\" | sed -n '1,2p'"], text=True, capture_output=True).stdout)

In [5]:
! nvidia-smi

Sun Oct 26 23:48:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:10:00.0 Off |                    0 |
| N/A   34C    P0             73W /  400W |   10285MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
! df -h

Filesystem            Size  Used Avail Use% Mounted on
devtmpfs              504G     0  504G   0% /dev
tmpfs                 504G  964K  504G   1% /dev/shm
tmpfs                 504G  178M  504G   1% /run
tmpfs                 504G     0  504G   0% /sys/fs/cgroup
/dev/mapper/vg1-root   32G   11G   22G  33% /
/dev/nvme0n1p2       1016M  218M  799M  22% /boot
/dev/nvme0n1p6        1.5T   16G  1.5T   2% /scratch
/dev/nvme0n1p1        599M  5.9M  594M   1% /boot/efi
/dev/mapper/vg1-var   8.0G  788M  7.3G  10% /var
/dev/mapper/vg1-tmp   100G  755M  100G   1% /tmp
home                  131T   19T  113T  15% /rhome
bigdata               3.8P  3.1P  693T  83% /bigdata


In [7]:
%pip install beir=='2.0.0'

Note: you may need to restart the kernel to use updated packages.


In [15]:
import torch
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(dev)

/rhome/znazi002/.conda/envs/pytorch_env/lib/python3.13/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


cuda


In [16]:
torch.__version__

'2.6.0+cu124'

In [17]:
import transformers

/rhome/znazi002/.conda/envs/pytorch_env/lib/python3.13/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [18]:
transformers.__version__

'4.55.4'

In [19]:
import accelerate

In [20]:
accelerate.__version__

'1.8.1'

In [21]:
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
print("Using dtype:", DTYPE)

torch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA A100-SXM4-80GB
Using dtype: torch.bfloat16


In [27]:
import torchvision

In [8]:
import os
import pathlib
import logging
from datetime import timedelta
from typing import List, Dict, Union, Tuple

import numpy as np
import torch
from torch import Tensor
import torch.distributed as dist
from tqdm import trange
from transformers import AutoTokenizer, AutoModel
from transformers.file_utils import PaddingStrategy

from beir import util, LoggingHandler
from beir.retrieval import models
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

import torch
import numpy as np
from typing import List, Dict
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES


# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.INFO,
                    handlers=[LoggingHandler()])
logger = logging.getLogger(__name__)


class ContrieverDenseRetriever:
    def __init__(self, model_path="facebook/contriever-msmarco"):
        """Initialize the Contriever model for both query and document encoding"""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load Model
        self.model = AutoModel.from_pretrained(model_path).to(self.device)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

    def mean_pooling(self, token_embeddings, mask):
        """Applies mean pooling to get sentence-level embeddings"""
        token_embeddings = token_embeddings.masked_fill(~mask[..., None].bool(), 0.)
        sentence_embeddings = token_embeddings.sum(dim=1) / mask.sum(dim=1)[..., None]
        return sentence_embeddings

    def encode_queries(self, queries: List[str], batch_size: int = 16, **kwargs) -> np.ndarray:
        """Encodes queries into dense embeddings"""
        query_embeddings = []

        with torch.no_grad():
            for start_idx in trange(0, len(queries), batch_size):
                encoded = self.tokenizer(
                    queries[start_idx:start_idx+batch_size], truncation=True, padding=True, return_tensors='pt', max_length=512
                ).to(self.device)

                outputs = self.model(**encoded)
                embeds = self.mean_pooling(outputs.last_hidden_state, encoded['attention_mask'])
                query_embeddings.append(embeds.cpu().numpy())

        return np.vstack(query_embeddings)

    def encode_corpus(self, corpus: List[Dict[str, str]], batch_size: int = 16, **kwargs) -> np.ndarray:
        """Encodes articles (title + abstract) into dense embeddings"""
        corpus_embeddings = []

        with torch.no_grad():
            for start_idx in trange(0, len(corpus), batch_size):
                titles = [row.get('title', '') for row in corpus[start_idx: start_idx + batch_size]]
                texts = [row.get('text', '') for row in corpus[start_idx: start_idx + batch_size]]

                encoded = self.tokenizer(
                    titles, texts, truncation='longest_first', padding=True, return_tensors='pt', max_length=512
                ).to(self.device)

                outputs = self.model(**encoded)
                embeds = self.mean_pooling(outputs.last_hidden_state, encoded['attention_mask'])
                corpus_embeddings.append(embeds.cpu().numpy())

        return np.vstack(corpus_embeddings)

/rhome/znazi002/.conda/envs/pytorch_env/lib/python3.13/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/rhome/znazi002/.conda/envs/pytorch_env/lib/python3.13/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [9]:
contriever = DRES(ContrieverDenseRetriever())
retriever_contriever = EvaluateRetrieval(contriever, score_function="dot") # or "cos_sim" for cosine similarity

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
# Download and load dataset
dataset = "nfcorpus" # dataset name
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)

In [11]:
# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

2025-10-26 23:49:52 - Loading Corpus...


  0%|          | 0/3633 [00:00<?, ?it/s]

2025-10-26 23:49:52 - Loaded 3633 TEST Documents.
2025-10-26 23:49:52 - Doc Example: {'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 6,011 participants die

In [12]:
import json
# load paraphrased dataset
with open(f"{dataset}_query_paraphrased_gpt4o.json", encoding='utf-8') as f:
    # Load the JSON data into a Python dictionary
    queries_para = json.load(f)

In [13]:
queries_p = {}
for q in queries_para:
  queries_p[q] = queries_para[q]['query_p']

In [14]:
results_contriever = retriever_contriever.retrieve(corpus, queries_p)

2025-10-26 23:50:23 - Encoding Queries...


100%|██████████| 3/3 [00:00<00:00,  5.97it/s]


2025-10-26 23:50:24 - Sorting Corpus by document length (Longest first)...
2025-10-26 23:50:24 - Encoding Corpus in batches... Warning: This might take a while!
2025-10-26 23:50:24 - Scoring Function: Dot Product (dot)
2025-10-26 23:50:24 - Encoding Batch 1/1...


100%|██████████| 29/29 [00:17<00:00,  1.65it/s]


In [15]:
# Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever_contriever.evaluate(qrels, results_contriever, retriever_contriever.k_values)

2025-10-26 23:51:55 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-10-26 23:51:55 - 

2025-10-26 23:51:55 - NDCG@1: 0.3669
2025-10-26 23:51:55 - NDCG@3: 0.3306
2025-10-26 23:51:55 - NDCG@5: 0.3142
2025-10-26 23:51:55 - NDCG@10: 0.2882
2025-10-26 23:51:55 - NDCG@100: 0.2725
2025-10-26 23:51:55 - NDCG@1000: 0.3667
2025-10-26 23:51:55 - 

2025-10-26 23:51:55 - MAP@1: 0.0459
2025-10-26 23:51:55 - MAP@3: 0.0723
2025-10-26 23:51:55 - MAP@5: 0.0864
2025-10-26 23:51:55 - MAP@10: 0.1008
2025-10-26 23:51:55 - MAP@100: 0.1283
2025-10-26 23:51:55 - MAP@1000: 0.1427
2025-10-26 23:51:55 - 

2025-10-26 23:51:55 - Recall@1: 0.0459
2025-10-26 23:51:55 - Recall@3: 0.0814
2025-10-26 23:51:55 - Recall@5: 0.1076
2025-10-26 23:51:55 - Recall@10: 0.1368
2025-10-26 23:51:55 - Recall@100: 0.2821
2025-10-26 23:51:55 - Recall@1000: 0.6219
2025-10-26 23:51:55 - 

2025-10-26 23:51:55 - P@1: 0.3808
2025-10-26 23:51:55

In [16]:
print(f"Model: Contriever; Dataset: {dataset} (paraphrased)")
print("-" * 150)
print(ndcg)
print(_map)
print(recall)
print(precision)

Model: Contriever; Dataset: nfcorpus (paraphrased)
------------------------------------------------------------------------------------------------------------------------------------------------------
{'NDCG@1': 0.36687, 'NDCG@3': 0.33062, 'NDCG@5': 0.31419, 'NDCG@10': 0.28821, 'NDCG@100': 0.27245, 'NDCG@1000': 0.36667}
{'MAP@1': 0.04593, 'MAP@3': 0.07234, 'MAP@5': 0.08641, 'MAP@10': 0.10076, 'MAP@100': 0.12834, 'MAP@1000': 0.14266}
{'Recall@1': 0.04593, 'Recall@3': 0.08135, 'Recall@5': 0.10763, 'Recall@10': 0.13679, 'Recall@100': 0.28206, 'Recall@1000': 0.6219}
{'P@1': 0.3808, 'P@3': 0.31269, 'P@5': 0.27492, 'P@10': 0.21765, 'P@100': 0.07288, 'P@1000': 0.02046}


In [17]:
import json
import pytrec_eval

# Use pytrec_eval to get per-query metrics
k_values = [10]
map_string = "map_cut." + ",".join([str(k) for k in k_values])
ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
recall_string = "recall." + ",".join([str(k) for k in k_values])
precision_string = "P." + ",".join([str(k) for k in k_values])

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
scores = evaluator.evaluate(results_contriever)

# Extract per-query metrics
per_query_metrics = {}
for query_id in scores.keys():
    per_query_metrics[query_id] = {
        "ndcg@10": scores[query_id]["ndcg_cut_10"],
        "recall@10": scores[query_id]["recall_10"],
        "map@10": scores[query_id]["map_cut_10"]
    }

# Calculate average metrics
avg_ndcg_10 = sum([m["ndcg@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_recall_10 = sum([m["recall@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_map_10 = sum([m["map@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)

# Print comparison
print("\n" + "="*80)
print("METRICS COMPARISON @ k=10")
print("="*80)
print(f"{'Metric':<20} {'Original':<20} {'Custom (Avg)':<20} {'Difference':<20}")
print("-"*80)
print(f"{'NDCG@10':<20} {ndcg['NDCG@10']:<20.4f} {avg_ndcg_10:<20.4f} {abs(ndcg['NDCG@10'] - avg_ndcg_10):<20.6f}")
print(f"{'Recall@10':<20} {recall['Recall@10']:<20.4f} {avg_recall_10:<20.4f} {abs(recall['Recall@10'] - avg_recall_10):<20.6f}")
print(f"{'MAP@10':<20} {_map['MAP@10']:<20.4f} {avg_map_10:<20.4f} {abs(_map['MAP@10'] - avg_map_10):<20.6f}")
print("="*80)

# Save to JSON file
output_filename = f"{dataset}_per_query_metrics_contriever_paraphrased.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print(f"\nPer-query metrics saved to {output_filename}")
print(f"Total queries processed: {len(per_query_metrics)}")


METRICS COMPARISON @ k=10
Metric               Original             Custom (Avg)         Difference          
--------------------------------------------------------------------------------
NDCG@10              0.2882               0.2882               0.000001            
Recall@10            0.1368               0.1368               0.000003            
MAP@10               0.1008               0.1008               0.000005            

Per-query metrics saved to nfcorpus_per_query_metrics_contriever_paraphrased.json
Total queries processed: 323


In [18]:
# Download and load dataset
dataset = "scifact" # dataset name
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)

In [19]:
# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

2025-10-26 23:55:20 - Loading Corpus...


  0%|          | 0/5183 [00:00<?, ?it/s]

2025-10-26 23:55:20 - Loaded 5183 TEST Documents.
2025-10-26 23:55:20 - Doc Example: {'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 vers

In [20]:
import json
# load paraphrased dataset
with open(f"{dataset}_query_paraphrased_gpt4o.json", encoding='utf-8') as f:
    # Load the JSON data into a Python dictionary
    queries_para = json.load(f)

In [21]:
queries_p = {}
for q in queries_para:
  queries_p[q] = queries_para[q]['query_p']

In [22]:
results_contriever = retriever_contriever.retrieve(corpus, queries_p)

2025-10-26 23:55:36 - Encoding Queries...


100%|██████████| 3/3 [00:00<00:00, 14.16it/s]


2025-10-26 23:55:36 - Sorting Corpus by document length (Longest first)...
2025-10-26 23:55:36 - Encoding Corpus in batches... Warning: This might take a while!
2025-10-26 23:55:36 - Scoring Function: Dot Product (dot)
2025-10-26 23:55:36 - Encoding Batch 1/1...


100%|██████████| 41/41 [00:23<00:00,  1.75it/s]


In [23]:
# Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever_contriever.evaluate(qrels, results_contriever, retriever_contriever.k_values)

2025-10-26 23:56:56 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-10-26 23:56:56 - 

2025-10-26 23:56:56 - NDCG@1: 0.5433
2025-10-26 23:56:56 - NDCG@3: 0.6295
2025-10-26 23:56:56 - NDCG@5: 0.6516
2025-10-26 23:56:56 - NDCG@10: 0.6747
2025-10-26 23:56:56 - NDCG@100: 0.7015
2025-10-26 23:56:56 - NDCG@1000: 0.7084
2025-10-26 23:56:56 - 

2025-10-26 23:56:56 - MAP@1: 0.5192
2025-10-26 23:56:56 - MAP@3: 0.5991
2025-10-26 23:56:56 - MAP@5: 0.6160
2025-10-26 23:56:56 - MAP@10: 0.6262
2025-10-26 23:56:56 - MAP@100: 0.6319
2025-10-26 23:56:56 - MAP@1000: 0.6321
2025-10-26 23:56:56 - 

2025-10-26 23:56:56 - Recall@1: 0.5192
2025-10-26 23:56:56 - Recall@3: 0.6892
2025-10-26 23:56:56 - Recall@5: 0.7433
2025-10-26 23:56:56 - Recall@10: 0.8118
2025-10-26 23:56:56 - Recall@100: 0.9360
2025-10-26 23:56:56 - Recall@1000: 0.9900
2025-10-26 23:56:56 - 

2025-10-26 23:56:56 - P@1: 0.5433
2025-10-26 23:56:56

In [24]:
print(f"Model: Contriever; Dataset: {dataset} (paraphrased)")
print("-" * 150)
print(ndcg)
print(_map)
print(recall)
print(precision)

Model: Contriever; Dataset: scifact (paraphrased)
------------------------------------------------------------------------------------------------------------------------------------------------------
{'NDCG@1': 0.54333, 'NDCG@3': 0.6295, 'NDCG@5': 0.65159, 'NDCG@10': 0.67473, 'NDCG@100': 0.70149, 'NDCG@1000': 0.70842}
{'MAP@1': 0.51917, 'MAP@3': 0.59915, 'MAP@5': 0.61597, 'MAP@10': 0.62622, 'MAP@100': 0.63185, 'MAP@1000': 0.63213}
{'Recall@1': 0.51917, 'Recall@3': 0.68922, 'Recall@5': 0.74333, 'Recall@10': 0.81178, 'Recall@100': 0.936, 'Recall@1000': 0.99}
{'P@1': 0.54333, 'P@3': 0.24889, 'P@5': 0.16533, 'P@10': 0.091, 'P@100': 0.0106, 'P@1000': 0.00112}


In [25]:
import json
import pytrec_eval

# Use pytrec_eval to get per-query metrics
k_values = [10]
map_string = "map_cut." + ",".join([str(k) for k in k_values])
ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
recall_string = "recall." + ",".join([str(k) for k in k_values])
precision_string = "P." + ",".join([str(k) for k in k_values])

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
scores = evaluator.evaluate(results_contriever)

# Extract per-query metrics
per_query_metrics = {}
for query_id in scores.keys():
    per_query_metrics[query_id] = {
        "ndcg@10": scores[query_id]["ndcg_cut_10"],
        "recall@10": scores[query_id]["recall_10"],
        "map@10": scores[query_id]["map_cut_10"]
    }

# Calculate average metrics
avg_ndcg_10 = sum([m["ndcg@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_recall_10 = sum([m["recall@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_map_10 = sum([m["map@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)

# Print comparison
print("\n" + "="*80)
print("METRICS COMPARISON @ k=10")
print("="*80)
print(f"{'Metric':<20} {'Original':<20} {'Custom (Avg)':<20} {'Difference':<20}")
print("-"*80)
print(f"{'NDCG@10':<20} {ndcg['NDCG@10']:<20.4f} {avg_ndcg_10:<20.4f} {abs(ndcg['NDCG@10'] - avg_ndcg_10):<20.6f}")
print(f"{'Recall@10':<20} {recall['Recall@10']:<20.4f} {avg_recall_10:<20.4f} {abs(recall['Recall@10'] - avg_recall_10):<20.6f}")
print(f"{'MAP@10':<20} {_map['MAP@10']:<20.4f} {avg_map_10:<20.4f} {abs(_map['MAP@10'] - avg_map_10):<20.6f}")
print("="*80)

# Save to JSON file
output_filename = f"{dataset}_per_query_metrics_contriever_paraphrased.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print(f"\nPer-query metrics saved to {output_filename}")
print(f"Total queries processed: {len(per_query_metrics)}")


METRICS COMPARISON @ k=10
Metric               Original             Custom (Avg)         Difference          
--------------------------------------------------------------------------------
NDCG@10              0.6747               0.6747               0.000003            
Recall@10            0.8118               0.8118               0.000002            
MAP@10               0.6262               0.6262               0.000001            

Per-query metrics saved to scifact_per_query_metrics_contriever_paraphrased.json
Total queries processed: 300


In [26]:
# Download and load dataset
dataset = "trec-covid" # dataset name
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)

In [27]:
# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

2025-10-26 23:58:18 - Loading Corpus...


  0%|          | 0/171332 [00:00<?, ?it/s]

2025-10-26 23:58:19 - Loaded 171332 TEST Documents.
2025-10-26 23:58:19 - Doc Example: {'text': 'OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, Saudi Arabia. METHODS: Patients with positive M. pneumoniae cultures from respiratory specimens from January 1997 through December 1998 were identified through the Microbiology records. Charts of patients were reviewed. RESULTS: 40 patients were identified, 33 (82.5%) of whom required admission. Most infections (92.5%) were community-acquired. The infection affected all age groups but was most common in infants (32.5%) and pre-school children (22.5%). It occurred year-round but was most common in the fall (35%) and spring (30%). More than three-quarters of patients (77.5%) had comorbidities. Twenty-four isolates (60%) were associated with pneumonia, 14 (35%) with upper respiratory tract 

In [28]:
import json
# load paraphrased dataset
with open(f"{dataset}_query_paraphrased_gpt4o.json", encoding='utf-8') as f:
    # Load the JSON data into a Python dictionary
    queries_para = json.load(f)

In [29]:
queries_p = {}
for q in queries_para:
  queries_p[q] = queries_para[q]['query_p']

In [30]:
results_contriever = retriever_contriever.retrieve(corpus, queries_p)

2025-10-26 23:58:31 - Encoding Queries...


100%|██████████| 1/1 [00:00<00:00, 30.53it/s]


2025-10-26 23:58:31 - Sorting Corpus by document length (Longest first)...
2025-10-26 23:58:31 - Encoding Corpus in batches... Warning: This might take a while!
2025-10-26 23:58:31 - Scoring Function: Dot Product (dot)
2025-10-26 23:58:31 - Encoding Batch 1/4...


100%|██████████| 391/391 [04:41<00:00,  1.39it/s]


2025-10-27 00:03:13 - Encoding Batch 2/4...


100%|██████████| 391/391 [03:46<00:00,  1.73it/s]


2025-10-27 00:06:59 - Encoding Batch 3/4...


100%|██████████| 391/391 [01:31<00:00,  4.25it/s]


2025-10-27 00:08:31 - Encoding Batch 4/4...


100%|██████████| 167/167 [00:06<00:00, 25.98it/s]


In [31]:
# Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever_contriever.evaluate(qrels, results_contriever, retriever_contriever.k_values)

2025-10-27 00:10:00 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-10-27 00:10:00 - 

2025-10-27 00:10:00 - NDCG@1: 0.5100
2025-10-27 00:10:00 - NDCG@3: 0.4864
2025-10-27 00:10:00 - NDCG@5: 0.4832
2025-10-27 00:10:00 - NDCG@10: 0.4527
2025-10-27 00:10:00 - NDCG@100: 0.3382
2025-10-27 00:10:00 - NDCG@1000: 0.3239
2025-10-27 00:10:00 - 

2025-10-27 00:10:00 - MAP@1: 0.0014
2025-10-27 00:10:00 - MAP@3: 0.0040
2025-10-27 00:10:00 - MAP@5: 0.0061
2025-10-27 00:10:00 - MAP@10: 0.0103
2025-10-27 00:10:00 - MAP@100: 0.0495
2025-10-27 00:10:00 - MAP@1000: 0.1299
2025-10-27 00:10:00 - 

2025-10-27 00:10:00 - Recall@1: 0.0014
2025-10-27 00:10:00 - Recall@3: 0.0044
2025-10-27 00:10:00 - Recall@5: 0.0070
2025-10-27 00:10:00 - Recall@10: 0.0128
2025-10-27 00:10:00 - Recall@100: 0.0798
2025-10-27 00:10:00 - Recall@1000: 0.3137
2025-10-27 00:10:00 - 

2025-10-27 00:10:00 - P@1: 0.5600
2025-10-27 00:10:00

In [32]:
print(f"Model: Contriever; Dataset: {dataset} (paraphrased)")
print("-" * 150)
print(ndcg)
print(_map)
print(recall)
print(precision)

Model: Contriever; Dataset: trec-covid (paraphrased)
------------------------------------------------------------------------------------------------------------------------------------------------------
{'NDCG@1': 0.51, 'NDCG@3': 0.48642, 'NDCG@5': 0.48318, 'NDCG@10': 0.45266, 'NDCG@100': 0.33818, 'NDCG@1000': 0.32393}
{'MAP@1': 0.00141, 'MAP@3': 0.00399, 'MAP@5': 0.00613, 'MAP@10': 0.01031, 'MAP@100': 0.04945, 'MAP@1000': 0.12994}
{'Recall@1': 0.00141, 'Recall@3': 0.0044, 'Recall@5': 0.007, 'Recall@10': 0.01278, 'Recall@100': 0.07982, 'Recall@1000': 0.3137}
{'P@1': 0.56, 'P@3': 0.54, 'P@5': 0.536, 'P@10': 0.488, 'P@100': 0.3518, 'P@1000': 0.15158}


In [33]:
import json
import pytrec_eval

# Use pytrec_eval to get per-query metrics
k_values = [10]
map_string = "map_cut." + ",".join([str(k) for k in k_values])
ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
recall_string = "recall." + ",".join([str(k) for k in k_values])
precision_string = "P." + ",".join([str(k) for k in k_values])

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
scores = evaluator.evaluate(results_contriever)

# Extract per-query metrics
per_query_metrics = {}
for query_id in scores.keys():
    per_query_metrics[query_id] = {
        "ndcg@10": scores[query_id]["ndcg_cut_10"],
        "recall@10": scores[query_id]["recall_10"],
        "map@10": scores[query_id]["map_cut_10"]
    }

# Calculate average metrics
avg_ndcg_10 = sum([m["ndcg@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_recall_10 = sum([m["recall@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_map_10 = sum([m["map@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)

# Print comparison
print("\n" + "="*80)
print("METRICS COMPARISON @ k=10")
print("="*80)
print(f"{'Metric':<20} {'Original':<20} {'Custom (Avg)':<20} {'Difference':<20}")
print("-"*80)
print(f"{'NDCG@10':<20} {ndcg['NDCG@10']:<20.4f} {avg_ndcg_10:<20.4f} {abs(ndcg['NDCG@10'] - avg_ndcg_10):<20.6f}")
print(f"{'Recall@10':<20} {recall['Recall@10']:<20.4f} {avg_recall_10:<20.4f} {abs(recall['Recall@10'] - avg_recall_10):<20.6f}")
print(f"{'MAP@10':<20} {_map['MAP@10']:<20.4f} {avg_map_10:<20.4f} {abs(_map['MAP@10'] - avg_map_10):<20.6f}")
print("="*80)

# Save to JSON file
output_filename = f"{dataset}_per_query_metrics_contriever_paraphrased.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print(f"\nPer-query metrics saved to {output_filename}")
print(f"Total queries processed: {len(per_query_metrics)}")


METRICS COMPARISON @ k=10
Metric               Original             Custom (Avg)         Difference          
--------------------------------------------------------------------------------
NDCG@10              0.4527               0.4527               0.000001            
Recall@10            0.0128               0.0128               0.000003            
MAP@10               0.0103               0.0103               0.000004            

Per-query metrics saved to trec-covid_per_query_metrics_contriever_paraphrased.json
Total queries processed: 50
